# MongoDB sampling estimates

Start the toolkit Docker MongoDB and initialize its `demo.business` fixture (see the local databases guide). This notebook reads that collection and temporarily creates a sample in `dbrtoolkit_samples`. It removes that sample afterward. Jupyter uses the async interface. Small samples here illustrate the API, not accurate estimates.

In [ ]:
from dbrtoolkit.mongodb import (
    AsyncMongoExecutor,
    AsyncMongoSampleEstimator,
    MongoCollectionRef,
    MongoFindQuery,
)

target = MongoCollectionRef("demo", "business")
query = MongoFindQuery(target, {"stars": {"$gte": 4}}, limit=1)
executor = AsyncMongoExecutor.from_uri("mongodb://localhost:27017")

## Full population versus a fresh sample
The find limit is deliberately one: estimation measures filter matches **before** limit. Omitting sampling options gives full-population evaluation.

In [ ]:
async with AsyncMongoSampleEstimator(executor, target) as estimator:
    exact = await estimator.estimate(query)

async with AsyncMongoSampleEstimator(executor, target, sample_ratio=0.5) as estimator:
    fresh = await estimator.estimate(query)

[(name, r.cardinality, r.selectivity, r.sample_matches, r.sample_size)
 for name, r in [("full", exact), ("sample", fresh)]]

## Reuse one sample
A persisted sample avoids drawing new documents for every query. The context manager prepares it and cleans it up, even if an estimate fails. Estimates from a tiny sample can differ substantially from full-population counts.

In [ ]:
async with AsyncMongoSampleEstimator(
    executor, target, sample_size=4, persist=True,
) as estimator:
    sample_target = estimator.sample_target
    first = await estimator.estimate(query)
    second = await estimator.estimate(query)
    print("Temporary collection:", sample_target)
    print(first)
    assert first == second

print("Sample removed:", estimator.sample_target is None)

## Explicit preparation and cleanup
You can also manage the lifetime yourself. Use one sample control only; oversized samples and ratios that round to zero issue warnings.

In [ ]:
estimator = AsyncMongoSampleEstimator(executor, target, sample_ratio=0.5)
try:
    await estimator.prepare()
    print(await estimator.estimate(query))
finally:
    await estimator.close()
    await executor.close()

## In an ordinary Python script
Use `MongoExecutor` and `MongoSampleEstimator`, `with` instead of `async with`, and omit `await`. The synchronous adapter uses the executor’s existing loop. Keep using async in Jupyter.

The population count is recorded at preparation; keep source data stable. Zero sample matches are not proof of zero population matches. Learned estimators can later implement the same estimation protocol without owning any sample.